> 이전 단계별 통합 실습 기록입니다. 앞으로의 기본 개발은 `mock_data_development.ipynb`, 실제 연결 준비는 `real_data_ready.ipynb`를 사용하세요. 이 노트북 일부 셀은 실제 Gemini 또는 DB에 접속할 수 있으므로 전체 실행하지 마세요.

# Agent ↔ Gemini API 연결 확인

목표: 임시 메뉴 6개에서 **4,000원 이하 커피를 모두 검색**합니다. 앞부분은 JSON 임시 데이터를 사용하며, 마지막 4절은 Docker 데모 DB와 Express까지 연결합니다. 실제 매장 데이터·대화 세션은 사용하지 않습니다.

1. 환경·데이터 준비 → 2. 오프라인 점검 → 3. 실제 API 실행.

`ai-server/.venv` 커널을 선택하세요. 기존 `requirements-notebook.txt`가 필요합니다. `.env`의 `GEMINI_API_KEY`, `GEMINI_MODEL`을 사용하며 키를 출력하지 않습니다. `RUN_LIVE=True`인 경우에만 API를 호출합니다.

이 실험은 API와 Agent의 연결 검증입니다. 필수 조건은 Python에도 명시하므로 자연어 조건 추출 정확도만을 평가하는 실험은 아닙니다.

In [ ]:
from pathlib import Path
import sys, json
cwd = Path.cwd().resolve()
AI_ROOT = next((p for base in (cwd, *cwd.parents) for p in (base, base / 'ai-server')
                if (p / 'app/agent/loop.py').is_file()), None)
if AI_ROOT is None:
    raise RuntimeError('coffee-station 또는 ai-server 안에서 실행하세요.')
sys.path.insert(0, str(AI_ROOT)) if str(AI_ROOT) not in sys.path else None

from app.repositories.mock import MockRepository
from app.repositories.base import SearchFilters
from app.agent.loop import run_turn
from app.llm.base import ScriptedModel, Reply, ToolCall
from app.llm.settings import read_gemini_settings
from app.llm.diagnostics import diagnose_model_error

repo = MockRepository(AI_ROOT / 'fixtures/api_smoke_catalog.json')
constraints = SearchFilters(max_price_krw=4000, category='COFFEE', sort_by='price')
message = '임시 메뉴에서 4000원 이하 COFFEE 메뉴를 빠짐없이 모두 찾아줘. search_menus 도구를 사용해.'
expected_ids = {'m1', 'm4', 'm5'}
for item in repo.search_menus(SearchFilters()):
    print(item.offer.id, item.offer.name, item.offer.price_krw, item.offer.category)
print('예상 정답: 에스프레소 3000원 / 아메리카노 3500원 / 디카페인 커피 4000원')
assert {m.offer.id for m in repo.search_menus(constraints)} == expected_ids

## 호출 흐름 표시와 판정

연결 PASS는 실제 모델이 검색 도구를 호출하고, 성공한 도구 결과를 다음 API 요청으로 전달받은 후 RESULT를 반환했을 때만 표시합니다. 메뉴 정답 일치는 별도로 검사합니다. 아래 출력 인자는 이 실습의 가상 조건에만 사용하세요.

In [ ]:
class ObservedModel:
    def __init__(self, delegate):
        self.delegate = delegate
        self.tool_result_roundtrip = False
        self.selected_menu_tool = False
    def generate(self, system, messages, tools, timeout_s):
        has_menu_result = any(
            m.get('role') == 'tool' and any(
                r['call'].name == 'search_menus' and r['response']['status'] == 'ok'
                for r in m['results'])
            for m in messages)
        print('→ Gemini: ' + ('검색 결과 전달' if has_menu_result else '요청 전달'), flush=True)
        reply = self.delegate.generate(system, messages, tools, timeout_s)
        if has_menu_result:
            self.tool_result_roundtrip = True
        for call in reply.calls:
            print('← 도구 호출:', call.name, json.dumps(call.arguments, ensure_ascii=False), flush=True)
            self.selected_menu_tool |= call.name == 'search_menus'
        if not reply.calls:
            print('← 모델 최종 응답 수신', flush=True)
        return reply

def report(result, observer, live=False):
    print(result.status, result.message)
    connected = (result.status == 'RESULT' and observer.selected_menu_tool
                 and observer.tool_result_roundtrip)
    print(('실제 API 연결' if live else '오프라인 흐름') + ':', 'PASS' if connected else '미확인/실패')
    actual = {m.offer.id for m in result.recommendations}
    print('메뉴 정답 검사:', 'PASS' if connected and actual == expected_ids else '미통과')
    for m in result.recommendations:
        print(m.offer.id, m.offer.name, m.offer.price_krw)
    print('LLM 요청 수:', result.llm_calls, '총 시간(ms):', result.elapsed_ms)
    for event in result.trace:
        print(event)

## 1. 오프라인 점검 — API 호출 없음

In [ ]:
fake = ObservedModel(ScriptedModel([
    Reply(calls=(ToolCall('search_menus', {'max_price_krw': 4000, 'category': 'COFFEE'}),)),
    Reply(text=json.dumps({'type':'RESULT','offer_ids':['m4','m1','m5']})),
]))
offline = run_turn(fake, repo, message, constraints=constraints, max_llm_calls=3)
report(offline, fake)
assert offline.status == 'RESULT' and fake.tool_result_roundtrip
assert {m.offer.id for m in offline.recommendations} == expected_ids

## 2. 실제 Gemini API 연결 테스트

아래 `RUN_LIVE=True`로 변경하고 실행하세요. 입력 대기는 없습니다. 최대 3회 모델 요청, 도구 3회, 턴 시간 예산 60초이며 자동 재시도하지 않습니다. 성공 여부가 출력되면 멈추세요. 504라면 도구 호출 이전인지 이후인지 출력으로 구분할 수 있습니다.

In [ ]:
RUN_LIVE = False
live_result = None
if RUN_LIVE:
    client = None
    try:
        from app.llm.gemini import GeminiModel
        key, model_id = read_gemini_settings(AI_ROOT)
        print('Gemini 설정 확인 완료', flush=True)
        client = GeminiModel(api_key=key, model=model_id)
        del key
        observed = ObservedModel(client)
        live_result = run_turn(observed, repo, message, constraints=constraints,
                               max_llm_calls=3, max_tool_calls=3, timeout_s=60)
        report(live_result, observed, live=True)
    except Exception as error:
        diagnostic, hint = diagnose_model_error(error)
        print('설정/초기화 실패:', hint, diagnostic)
        print('GEMINI_API_KEY와 GEMINI_MODEL 설정 및 설치 환경을 확인하세요.')
    finally:
        if client is not None:
            client.close()
else:
    print('API 호출 없음. 실제 테스트는 RUN_LIVE=True로 바꿔 실행하세요.')

## 결과 해석

- **실제 API 연결 PASS + 메뉴 정답 PASS**: 이번 목표 완료.
- **연결 PASS + 메뉴 정답 미통과**: API 연결은 정상이며 모델의 선택 결과를 점검해야 합니다.
- **도구 호출 출력 없이 HTTP 504**: 첫 모델 응답 실패. Python 메뉴 검색은 아직 실행되지 않았습니다.
- **도구 호출 후 실패**: trace에서 도구 성공 여부와 다음 모델 요청을 확인하세요.

코드가 수정된 뒤에는 커널을 재시작하세요. 모델명만 .env에서 바꿨다면 실제 테스트 셀만 재실행하면 됩니다. 노트북 공유 전 출력을 지우세요.

## 4. Express → Python → Gemini → 데모 MySQL 통합 테스트

위의 직접 Agent 테스트와 별도로 실행합니다. **첫 환경 준비 셀을 실행한 후 이 절로 바로 이동해도 됩니다.**
임시 매장 1개·메뉴 6개는 `sql/demo_seed.sql`에 있고 데모 DB에만 저장됩니다.
아래 정답은 4000원 이하 COFFEE 3개입니다. LATTE는 별도 카테고리입니다.

필요: Docker `coffee-mysql` 실행 및 데모 시드, Node.js와 backend 의존성,
Python `requirements.txt`, `requirements-notebook.txt`, `requirements-mysql.txt` 설치.
DB 준비/삭제 방법은 `app/repositories/README.md`를 참고하세요.
API·DB 키는 .env에서 읽습니다. 이 절은 18081/18082 포트를 사용하고
테스트 서버를 자동 종료합니다. 실제 API 할당량은 `RUN_HTTP_TEST=True`일 때만 사용합니다.

코드 업데이트 후에는 커널을 재시작하고 첫 준비 셀부터 실행하세요. 이전 클래스를 부분 reload하지 마세요.


In [ ]:
# 먼저 DB 연결만 점검합니다. Gemini 호출은 없습니다.
from app.repositories.factory import create_repository
from app.repositories import factory as repository_factory
import inspect
if 'option_name' not in inspect.signature(repository_factory.MenuOffer).parameters:
    raise RuntimeError(
        '커널에 이전 MenuOffer 정의가 남아 있습니다. 커널을 재시작하고 '
        '첫 환경 준비 셀부터 다시 실행하세요. DB를 재생성할 필요는 없습니다.'
    )
sql_repo = create_repository('mysql-demo', cafe_ids=(990001,))
for item in sql_repo.search_menus(SearchFilters(limit=10, sort_by='price')):
    print(item.offer.id, item.offer.name, item.offer.price_krw, item.offer.category)
sql_expected = ['992004', '992001', '992005']
assert [m.offer.id for m in sql_repo.search_menus(
    SearchFilters(max_price_krw=4000, category='COFFEE', sort_by='price'))] == sql_expected
print('데모 DB 및 예상 정답: PASS')


In [ ]:
# 실제 요청: 노트북 → Express → Python → Gemini → MySQL → 추천 응답
RUN_HTTP_TEST = False  # 준비 후 True로 변경
if RUN_HTTP_TEST:
    import os, shutil, subprocess
    npm = shutil.which('npm.cmd' if os.name == 'nt' else 'npm')
    if not npm:
        raise RuntimeError('Node.js/npm 설치를 확인하세요.')
    subprocess.run([npm, 'run', 'build'], cwd=AI_ROOT.parent / 'backend', check=True, timeout=60)
    completed = subprocess.run(
        [sys.executable, str(AI_ROOT / 'scripts/check_recommendation_http.py'), '--express'],
        cwd=AI_ROOT, check=False, capture_output=True, text=True,
        encoding='utf-8', errors='replace', timeout=150,
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError('통합 테스트 실패: 위 상태와 DB/서버 설정을 확인하세요.')
else:
    print('준비 완료 후 RUN_HTTP_TEST=True로 변경하세요. 실제 API는 호출하지 않았습니다.')


## 5. 경계 조건·오류 처리 평가

첫 준비 셀 이후 실행할 수 있습니다. 자동 테스트는 실제 Gemini를 호출하지 않습니다.
가격 경계값/빈 결과/거리순/조건 완화 차단/DB 및 모델 오류를 검증합니다.
거리 테스트는 서로 다른 좌표의 임시 매장 2개를 메모리에만 생성합니다.
실제 데모 DB는 매장 1개이므로 매장 간 거리 순위를 평가할 수 없습니다.

실제 3개 시나리오 재검증은 4절의 build 실행 후 `RUN_CASE_TEST=True`로 실행하세요.
4000원→3개, 3999원→2개, 2999원→0개 및 RESULT 또는 QUESTION을 기대합니다.
실제 Gemini 할당량을 사용하며, 실패하면 다음 시나리오를 실행하지 않습니다.


In [ ]:
import subprocess
checked = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=AI_ROOT, capture_output=True, text=True, timeout=60)
print(checked.stdout, checked.stderr)
checked.check_returncode()


In [ ]:
RUN_CASE_TEST = False
if RUN_CASE_TEST:
    import os
    checked = subprocess.run(
        [sys.executable, 'scripts/check_recommendation_http.py', '--express', '--cases'],
        cwd=AI_ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace',
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8'}, timeout=300)
    print(checked.stdout, checked.stderr)
    checked.check_returncode()
else:
    print('실제 3개 시나리오를 실행하려면 RUN_CASE_TEST=True로 변경하세요.')


## 6. 자연어 조건 추출 (미확정 제안)

코드 변경 후 **커널을 재시작하고 첫 환경 준비 셀**을 실행하세요.
문장에서 가격·종류·정렬을 추출합니다. 추출 결과는 자동으로 추천 API에 전달하지 않습니다.
`unresolved`에 미지원 조건이나 추가 질문이 있으면 먼저 해결해야 합니다.
비어 있어도 원문 근거(`evidence`)와 추출값을 사람이 확인해야 합니다.
현재 단계는 확인 화면/확인 상태 저장/API 연동을 포함하지 않습니다.

- 이하: 해당 가격 포함 / 미만: 정수 원화에서 1원 차감
- 라떼는 LATTE이며 COFFEE와 구분
- '저렴한 순'만으로 예산을 생성하지 않음
- 단맛·디카페인·공부 목적·거리 기준 등은 확인 항목으로 남김

두 실행 스위치는 기본 False입니다. 실제 호출은 Gemini 할당량을 사용합니다.


In [ ]:
from app.agent.extraction import extract_proposal
from app.llm.gemini import GeminiModel

EXTRACTION_MESSAGE = '4천 원 이하 커피를 저렴한 순으로 추천해줘.'
RUN_EXTRACTION = False
proposal = None  # 이전 실행의 제안을 재사용하지 않음
if RUN_EXTRACTION:
    key, model_id = read_gemini_settings(AI_ROOT)
    extraction_model = GeminiModel(api_key=key, model=model_id)
    try:
        proposal = extract_proposal(extraction_model, EXTRACTION_MESSAGE)
        print('사용자 확인 전 제안 — 추천 검색은 실행하지 않았습니다.')
        print(proposal.model_dump_json(indent=2))
        print('추가 질문 필요' if proposal.unresolved else '추출 조건을 원문과 비교해 확인하세요.')
    finally:
        extraction_model.close()
else:
    print('RUN_EXTRACTION=True로 변경하면 위 문장을 실제로 추출합니다.')


In [ ]:
RUN_EXTRACTION_EVALUATION = False
if RUN_EXTRACTION_EVALUATION:
    import os, subprocess
    checked = subprocess.run(
        [sys.executable, 'scripts/evaluate_extraction.py'], cwd=AI_ROOT,
        capture_output=True, text=True, encoding='utf-8', errors='replace',
        env={**os.environ, 'PYTHONIOENCODING': 'utf-8'}, timeout=180)
    print(checked.stdout, checked.stderr)
    checked.check_returncode()
else:
    print('5개 문장 평가: RUN_EXTRACTION_EVALUATION=True로 실행합니다.')


## 7. 조건 확인·수정 → 추천

실제 사용자 흐름을 시험하려면 별도 터미널 2개에서 서버를 실행하세요.
루트 .env의 INTERNAL_API_TOKEN을 두 서버가 동일하게 사용해야 합니다.

**터미널 1 (ai-server)**
```powershell
$env:AGENT_REPOSITORY='mysql-demo'
$env:AGENT_CAFE_IDS='990001'
.\.venv\Scripts\python.exe -m uvicorn app.main:app --host 127.0.0.1 --port 8000
```
**터미널 2 (backend)**
```powershell
$env:AI_SERVER_URL='http://127.0.0.1:8000'
$env:HOST='127.0.0.1'
$env:PORT='8080'
npm run dev
```
Python의 ai-server/.env에 별도 토큰이 있으면 루트와 같게 맞추세요. 데모 DB는 실행 중이어야 합니다.
아래 준비 셀은 추출만 합니다. 결과를 읽고 수정한 다음 확인 셀에서 명시적으로 동의하세요.
미해결 조건은 수정 본문에서 삭제하는 것으로 해결할 수 없습니다. 문장을 수정하여 다시 준비해야 합니다.
예: 단맛이 중요하면 현 데이터로는 확인 불가. 사용자가 동의할 때만 그 조건을 뺀 문장으로 재요청합니다.
제안은 10분/1회 유효하며 서버 재시작 시 사라집니다. 검색 실패 후에도 새 준비가 필요합니다.


In [ ]:
import httpx, copy
RUN_PREPARE = False
prepared = None
confirmed_constraints = None
if RUN_PREPARE:
    with httpx.Client(timeout=45, trust_env=False) as client:
        response = client.post('http://127.0.0.1:8080/api/recommendations/prepare',
            json={'message': '4천 원 이하 커피를 저렴한 순으로 추천해줘.'})
    if response.is_error:
        print('HTTP:', response.status_code, '오류:', response.json().get('message', '응답 처리 실패'))
    response.raise_for_status()
    prepared = response.json()['data']
    confirmed_constraints = copy.deepcopy(prepared['suggested_constraints'])
    print('추출 모드:', prepared.get('extraction_mode'), 'LLM 호출:', prepared.get('llm_calls'))
    print('상태:', prepared['status'])
    print('추출 근거:', prepared['proposal']['evidence'])
    print('추가 확인:', prepared['proposal']['unresolved'])
    print('확인할 조건:', confirmed_constraints)
    print(prepared['notice'])
else:
    print('RUN_PREPARE=True로 준비합니다. 추천 검색은 하지 않습니다.')


In [ ]:
# 필요하면 먼저 수정: confirmed_constraints['max_price_krw'] = 3500
USER_CONFIRMED = False  # 표시된 조건을 확인한 뒤 True
if USER_CONFIRMED:
    if prepared is None or prepared['status'] != 'AWAITING_CONFIRMATION':
        raise RuntimeError('먼저 준비하고 미해결 조건을 해결하세요.')
    print('확정하여 전송할 조건:', confirmed_constraints)
    with httpx.Client(timeout=100, trust_env=False) as client:
        response = client.post('http://127.0.0.1:8080/api/recommendations/confirm', json={
            'proposal_id': prepared['proposal_id'], 'confirmed': True,
            'constraints': confirmed_constraints})
    if response.is_error:
        print('HTTP:', response.status_code, '오류:', response.json().get('message', '응답 처리 실패'))
    response.raise_for_status()
    result = response.json()['data']
    print(result['status'], result['message'])
    for match in result['recommendations']:
        print(match['offer']['name'], match['offer']['price_krw'])
else:
    print('사용자 확인 전: 추천을 실행하지 않았습니다.')


서버 자동 기동/종료를 포함한 회귀 테스트는 backend에서 `npm run build` 후
ai-server에서 `.venv/Scripts/python.exe scripts/check_recommendation_http.py --express --confirm`.
이 명령은 고정된 테스트 조건을 자동 확인하며 실제 사용자 동의 UI를 검증하는 것은 아닙니다.


## 8. 확정 조건 직접 추천 — LLM 호출 0회

첫 준비 셀 후 실행합니다. JSON 임시 메뉴 6개를 사용하므로 Docker/API 키가 필요 없습니다.
서버 기본값도 RECOMMENDATION_MODE=direct입니다. 서버 재시작 후 적용됩니다.
자연어 prepare는 여전히 Gemini를 1회 호출하며, confirm의 검색·정렬은 Python이 처리합니다.
Agent Loop 실습을 하려면 서버 설정 RECOMMENDATION_MODE=agent로 변경하세요.
직접 추천은 자연어를 해석하지 않습니다. 확정 조건만 적용합니다.


In [ ]:
from app.agent.direct import recommend_direct
local_repo = MockRepository(AI_ROOT / 'fixtures/api_smoke_catalog.json')
for budget, expected in [(4000, ['m4', 'm1', 'm5']), (3999, ['m4', 'm1']), (2999, [])]:
    result = recommend_direct(local_repo, SearchFilters(
        max_price_krw=budget, category='COFFEE', sort_by='price'))
    assert [m.offer.id for m in result.recommendations] == expected
    assert result.llm_calls == 0
    print(budget, result.status, 'LLM 호출:', result.llm_calls)
    print(result.message)
    for match in result.recommendations:
        print(match.offer.name, match.offer.price_krw)


## 9. API 비용 없는 서버 개발 모드

Python 서버를 끄고 ai-server 터미널에서 다음 설정으로 재실행합니다.
```powershell
$env:EXTRACTION_MODE='mock'
$env:RECOMMENDATION_MODE='direct'
$env:AGENT_REPOSITORY='mock'
.\.venv\Scripts\python.exe -m uvicorn app.main:app --host 127.0.0.1 --port 8000
```
Express는 기존 방식으로 실행합니다. 내부 인증 토큰은 여전히 두 서버에서 일치해야 합니다.
7절의 준비·확인 셀을 사용하면 Gemini와 Docker 없이 전체 흐름을 시험할 수 있습니다.
기본 mock Repository의 임시 데이터는 데모 MySQL 메뉴와 달라 결과 ID/개수가 다를 수 있습니다.
추출 모드 mock, LLM 호출 0을 확인하세요.

지원 문장(정확히 입력):
- 4천 원 이하 커피를 저렴한 순으로 추천해줘.
- 5천 원 미만 라떼를 추천해줘.
- 근처에서 달지 않은 커피를 가까운 순으로 찾아줘.

세 번째 문장과 미등록 문장은 추가 확인 상태로 반환합니다. 일반 자연어 해석 기능이 아닙니다.
실제 추출은 EXTRACTION_MODE=gemini로 변경 후 서버를 재시작합니다.
이 설정은 서버에 적용됩니다. 앞부분에서 GeminiModel을 직접 사용하는 셀은 실제 API를 호출합니다.


In [ ]:
# 외부 API/DB 없이 prepare → confirm 자동 검증
import subprocess
checked = subprocess.run([sys.executable, '-m', 'unittest', 'discover',
    '-s', 'tests', '-p', 'test_mock_flow.py', '-v'], cwd=AI_ROOT,
    capture_output=True, text=True, timeout=60)
print(checked.stdout, checked.stderr)
checked.check_returncode()


## 10. 리뷰 저장형 요약 준비 (API 호출 없음)
가상 리뷰의 근거 발췌를 저장합니다. 자연어 종합 요약이 아닌 모의 결과입니다.
동일 내용이면 재사용, 수정/삭제/숨김이면 갱신합니다. 테스트 파일은 종료 시 자동 삭제됩니다.


In [ ]:
from app.review_summary import Review, SummaryService, MockReviewSummarizer
from tempfile import TemporaryDirectory
from dataclasses import replace
with TemporaryDirectory() as folder:
    service = SummaryService(Path(folder) / 'review-summary.sqlite', MockReviewSummarizer())
    reviews = [Review(1, 990001, '평일에는 조용하고 좌석이 넓어요.'),
               Review(2, 990001, '주말에는 대화 소리가 커요.'),
               Review(3, 990001, '숨겨진 리뷰', status='HIDDEN')]
    summary, refreshed = service.refresh(990001, reviews)
    print('최초 생성:', refreshed, summary)
    assert not service.refresh(990001, reviews)[1]
    reviews[0] = replace(reviews[0], deleted_at='2026-11-09')
    summary, refreshed = service.refresh(990001, reviews)
    assert summary['source_review_ids'] == [2]
    print('삭제 반영:', refreshed, summary)
print('PASS: 임시 저장 파일 자동 삭제 완료')
